In [ ]:
from transformers import Qwen3Model

model = Qwen3Model.from_pretrained("Qwen/Qwen3-72B-Instruct")

In [ ]:
# RMSNorm
import torch
import torch.nn as nn
from torch import nn

class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
        self.variance_epsilon = eps
    
    def _norm(self, x):
        return x * torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
    
    def forward(self, x):
        output = self._norm(x.float())
        return output * self.weight

rms_norm = RMSNorm(dim=4, eps=1e-6)

x = torch.tensor([[
    [1.0, 2.0, 3.0, 4.0],
    [2.0, 3.0, 4.0, 5.0],
    [3.0, 4.0, 5.0, 6.0]],

    [[-1.0, -2.0, -3.0, -4.0],
    [-2.0, -3.0, -4.0, -5.0],
    [-3.0, -4.0, -5.0, -6.0]]])

print(x.shape)

output = rms_norm(x)
print(f'RMSNorm 输出shape: {output.shape}')
print(f'RMSNorm[0][0] 输出: {output[0][0]}')

torch.Size([2, 3, 4])
RMSNorm 输出shape: torch.Size([2, 3, 4])
RMSNorm[0][0] 输出: tensor([0.3651, 0.7303, 1.0954, 1.4606], grad_fn=<SelectBackward0>)


In [ ]:
x_sample = x[0][0].float()
rms = torch.sqrt(x_sample.pow(2).mean() + 1e-6)
manual_calculation_result = x_sample / rms

print(f'手动计算结果: {manual_calculation_result}')
print(f'RMSNorm 输出: {output[0][0]}')

print('Results comparison:', torch.allclose(manual_calculation_result, output[0][0], atol=1e-4))



In [ ]:
from transformers import Qwen3Model

model = Qwen3Model.from_pretrained("Qwen/Qwen3-72B-Instruct")

In [87]:
import math

class Qwen3Config:
    def __init__(self):
        self.hidden_size = 1024
        self.num_attention_heads = 16
        self.num_key_value_heads = 8
        self.head_dim = 128
        self.rms_norm_eps = 1e-6
        self.layer_types = ["sliding_attention"] * 12
        self.sliding_window = 1024
        self._attn_implementation = "eager"
        self.vocab_size = 10000
        self.num_hidden_layers = 8
        self.attention_bias = False
        self.attention_dropout = 0.0
        self.intermediate_size = 128
        

In [60]:
def apply_rotary_pos_emb(q, k, cos, sin):
    print(f'q shape: {q.shape}')
    print(f'k shape: {k.shape}')
    print(f'cos shape: {cos.shape}')
    print(f'sin shape: {sin.shape}')
    def rotate_half(x):
        x1, x2 = x.chunk(2, dim=-1)
        return torch.cat((-x2, x1), dim=-1)
    
    q_embed = q * cos + rotate_half(q) * sin
    k_embed = k * cos + rotate_half(k) * sin
    return q_embed, k_embed


In [61]:
class Qwen3RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        # 针对最后一位 head_dim 进行 norm
        variance = x.pow(2).mean(-1, keepdim=True)
        x = x * torch.rsqrt(variance + self.eps)
        return self.weight * x

In [82]:
class Qwen3Attention(nn.Module):
    def __init__(self, config: Qwen3Config, layer_idx: int):
        super().__init__()
        self.config = config
        self.layer_idx = layer_idx
        self.num_heads = config.num_attention_heads
        self.n_kv_head = config.num_key_value_heads
        self.head_dim = config.head_dim
        # self.num_kv_group = self.n_kv_head // self.num_heads
        #!!!! 写反了
        self.num_kv_group = self.num_heads // self.n_kv_head
        self.scaling = config.head_dim ** -0.5
        
        self.q_proj = nn.Linear(config.hidden_size, self.num_heads * self.head_dim, bias=config.attention_bias)
        self.k_proj = nn.Linear(config.hidden_size, self.n_kv_head * self.head_dim, bias=config.attention_bias)
        self.v_proj = nn.Linear(config.hidden_size, self.n_kv_head * self.head_dim, bias=config.attention_bias)
        
        self.o_proj = nn.Linear(self.num_heads * self.head_dim, config.hidden_size, bias=config.attention_bias)
        
        self.q_norm = Qwen3RMSNorm(self.head_dim, config.rms_norm_eps)
        self.k_norm = Qwen3RMSNorm(self.head_dim, config.rms_norm_eps)

    def forward(self, hidden_states, cos, sin, attn_mask=None):

        batch_size, seq_len, _ = hidden_states.shape

        query_states = self.q_proj(hidden_states).view(batch_size, seq_len, self.num_heads, self.head_dim)
        key_states = self.k_proj(hidden_states).view(batch_size, seq_len, self.n_kv_head, self.head_dim)
        value_states = self.v_proj(hidden_states).view(batch_size, seq_len, self.n_kv_head, self.head_dim)

        query_states = self.q_norm(query_states).transpose(1, 2)
        key_states = self.k_norm(key_states).transpose(1, 2)
        value_states = value_states.transpose(1, 2)

        print(f'key_states shape: {key_states.shape}')
        
        query_states, key_states = apply_rotary_pos_emb(query_states, key_states, cos, sin)

        print(f'key_states shape: {key_states.shape}, {self.num_kv_group}')
        
        key_states = torch.repeat_interleave(key_states, dim=1, repeats=self.num_kv_group)
        print(f'key_states shape: {key_states.shape}!')
        value_states = torch.repeat_interleave(value_states, dim=1, repeats=self.num_kv_group)



        attn_weights = torch.matmul(query_states, key_states.transpose(2,3)) * self.scaling

        if attn_mask is not None:
            attn_weights = attn_weights + attn_mask

        attn_weights = torch.softmax(attn_weights, dim=-1).to(query_states.dtype)
        attn_output = torch.matmul(attn_weights, value_states)

        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, -1)

        attn_output = self.o_proj(attn_output)

        return attn_output, attn_weights
        
        
        
        
        

In [83]:
def run_example():
    config = Qwen3Config()
    model = Qwen3Attention(config, layer_idx=0)

    batch_size = 64
    seq_len = 256
    hidden_size = config.hidden_size
    head_dim = config.head_dim

    hidden_states = torch.randn(batch_size, seq_len, hidden_size)

    cos = torch.ones(1, 1, seq_len, head_dim)
    sin = torch.zeros(1, 1, seq_len, head_dim)

    attn_mask = torch.full((seq_len, seq_len), float("-inf"))
    attn_mask = torch.triu(attn_mask, diagonal=1)

    output, weights = model(hidden_states, cos, sin, attn_mask)

    print(f'input shape: {hidden_states.shape}')
    print(f'output shape: {output.shape}')
    print(f'weights shape: {weights.shape}')
    
run_example()

key_states shape: torch.Size([64, 8, 256, 128])
q shape: torch.Size([64, 16, 256, 128])
k shape: torch.Size([64, 8, 256, 128])
cos shape: torch.Size([1, 1, 256, 128])
sin shape: torch.Size([1, 1, 256, 128])
key_states shape: torch.Size([64, 8, 256, 128]), 2
key_states shape: torch.Size([64, 16, 256, 128])!
input shape: torch.Size([64, 256, 1024])
output shape: torch.Size([64, 256, 1024])
weights shape: torch.Size([64, 16, 256, 256])


In [30]:
import torch.nn.functional as F

class Qwen3MLP(nn.Module):
    def __init__(self, config: Qwen3Config):
        super().__init__()
        self.config = config
        self.intermediate_size = config.intermediate_size
        self.gate_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=config.attention_bias)
        self.up_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=config.attention_bias)
        self.down_proj = nn.Linear(config.intermediate_size, config.hidden_size, bias=config.attention_bias)
        
        self.act_fn = F.silu
    
    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))
        
        
        

In [85]:
class Qwen3DecoderLayer(nn.Module):
    def __init__(self, config: Qwen3Config, layer_idx: int):
        super().__init__()
        self.hidden_size = config.hidden_size
        self.self_attn = Qwen3Attention(config, layer_idx)
        self.mlp = Qwen3MLP(config)

        self.input_layernorm = RMSNorm(config.hidden_size, config.rms_norm_eps)
        self.post_attention_layernorm = RMSNorm(config.hidden_size, config.rms_norm_eps)

    def forward(self, hidden_states, attention_mask=None, position_embeddings=None):
        residual = hidden_states
        hidden_states = self.input_layernorm(hidden_states)
        
        hidden_states, attn_weights = self.self_attn(hidden_states, cos=position_embeddings[0], sin=position_embeddings[1], attn_mask=attention_mask)

        hidden_states = residual + hidden_states
        residual = hidden_states
        hidden_states = self.post_attention_layernorm(hidden_states)

        hidden_states = self.mlp(hidden_states)
        hidden_states = residual + hidden_states

        return hidden_states, attn_weights
        
        

In [88]:
def run_decoder_example():
    # 1. 基础配置

    config = Qwen3Config()
    layer = Qwen3DecoderLayer(config, layer_idx=0)
    
    # 2. 模拟输入数据
    batch_size = 1
    seq_len = 8
    hidden_states = torch.randn(batch_size, seq_len, config.hidden_size)

    # 3. 构造 RoPE (cos, sin)
    # 实际模型中这些是从 RoPE Embedding 类生成的，这里手动构造对应维度的 Tensor
    cos = torch.ones(batch_size, 1, seq_len, config.head_dim)
    sin = torch.zeros(batch_size, 1, seq_len, config.head_dim)
    position_embeddings = (cos, sin)

    # 4. 前向传播
    with torch.no_grad():
        output, weights = layer(
            hidden_states=hidden_states,
            position_embeddings=position_embeddings
        )

    print(f"Decoder Layer 输入形状: {hidden_states.shape}")
    print(f"Decoder Layer 输出形状: {output.shape}")
    print("--- 运行成功 ---")


run_decoder_example()

key_states shape: torch.Size([1, 8, 8, 128])
q shape: torch.Size([1, 16, 8, 128])
k shape: torch.Size([1, 8, 8, 128])
cos shape: torch.Size([1, 1, 8, 128])
sin shape: torch.Size([1, 1, 8, 128])
key_states shape: torch.Size([1, 8, 8, 128]), 2
key_states shape: torch.Size([1, 16, 8, 128])!
Decoder Layer 输入形状: torch.Size([1, 8, 1024])
Decoder Layer 输出形状: torch.Size([1, 8, 1024])
--- 运行成功 ---
